In [7]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1', on_bad_lines='skip', engine='python')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    14675
negative    14659
Name: count, dtype: int64


In [8]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
29329,About a year ago I finally gave up on American...,positive
29330,"OK, people, honestly... this gotta be one of t...",negative
29331,To me movies and acting is all about telling a...,positive
29332,Like the Arabian Nights this film plays with s...,positive


In [9]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})

In [11]:
from numpy import divide
import re
negation_words=[
    "not good","not bad","not great","don't like","didn't like","never liked","wasn't good","isn't good","no good"
]

def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z\s']","",text)

  #convert negations into single tokens
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [12]:
df['review']=df['review'].apply(clean_text)

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42)

In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
vocab_size=20000
max_len=250
tokenizer=Tokenizer(num_words=vocab_size,oov_token="<00V") #out of vocabulary
tokenizer.fit_on_texts(X_train)
X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)
X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post') #highest len onum chyilla,backi ellam zeroes itt adjust chyum.post means backil zeroes.
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [21]:
#LSTM(long short term memory)-advanced type of RNN
#forget gate-what to forget
#input gate-what to store
#cell state-in memory
#output gate-what to return
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout
model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
history=model.fit(X_train_pad,y_train,epochs=5,batch_size=64,validation_split=0.2)

Epoch 1/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 275s 936ms/step - accuracy: 0.8984 - loss: 0.2691 - val_accuracy: 0.8475 - val_loss: 0.4414
Epoch 2/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 269s 916ms/step - accuracy: 0.9364 - loss: 0.1823 - val_accuracy: 0.8515 - val_loss: 0.4319
Epoch 3/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 253s 860ms/step - accuracy: 0.9560 - loss: 0.1322 - val_accuracy: 0.8419 - val_loss: 0.4752
Epoch 4/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 242s 823ms/step - accuracy: 0.9685 - loss: 0.0986 - val_accuracy: 0.8451 - val_loss: 0.5375
Epoch 5/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 242s 824ms/step - accuracy: 0.9789 - loss: 0.0710 - val_accuracy: 0.8477 - val_loss: 0.5821


In [27]:
loss,acc=model.evaluate(X_test_pad,y_test)
print("Test accuracy:",acc)

184/184 ━━━━━━━━━━━━━━━━━━━━ 19s 102ms/step - accuracy: 0.8509 - loss: 0.5843
Test accuracy: 0.8508607745170593


In [28]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview",review)
  print("Score:",prediction)
  if prediction>=0.5:
    print("sentiment: Positive")
  else:
    print("sentiment: Negative")

In [29]:
predict_sentiment("This movie was absolutely amazing and I loved it very much")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 759ms/step

Review this movie was absolutely amazing and i loved it very much
Score: 0.9925886
sentiment: Positive


In [30]:
predict_sentiment("This movie was not good and I didn't like it at all")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step

Review this movie was not_good and i didn't_like it at all
Score: 0.36225215
sentiment: Negative


In [31]:
predict_sentiment("This movie is very entertaining! ")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step

Review this movie is very entertaining 
Score: 0.9228133
sentiment: Positive
